## ↓のデータフレーム変換関数を定義したセルを実行済みの状態で進めてください

In [ ]:
import requests
import time
import pandas as pd

def gen_boj_dataframe(
    db:str,
    codes:list[str],
    startdate:str,
    enddate:str) -> pd.DataFrame:
    url = "https://www.stat-search.boj.or.jp/api/v1/getDataCode"
    params = {
        "DB": db,
        "CODE": ",".join(codes),
        "FORMAT": "JSON",
        "LANG":"JP",
        "STARTDATE": startdate,
        "ENDDATE": enddate
        }

    # ページネーションで最後のページを取得するまで、繰り返し処理を実行
    # Web APIからの取得データを格納
    all_results = []

    while True:
        # Web APIへのリクエストと例外処理。エラー時は空のデータフレームを返して終了
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            response_data = response.json()
        except requests.exceptions.Timeout:
            print("Web APIから30秒以内に応答がありませんでした。")
            return pd.DataFrame()
        except requests.exceptions.ConnectionError:
            print("Web APIに接続できませんでした。")
            return pd.DataFrame()
        except requests.exceptions.HTTPError as error:
            print(f"HTTPエラーが発生しました: {error}")

            content_type = response.headers.get("Content-Type", "")
            if "application/json" in content_type:
                error_data = response.json()
                print(error_data["MESSAGE"])
            return pd.DataFrame()
        except requests.exceptions.JSONDecodeError:
            print("レスポンスをJSONとして読み込めませんでした。")
            return pd.DataFrame()
        else:
            if response_data["MESSAGEID"] == "M181030I":
                print(response_data["MESSAGE"])
                return pd.DataFrame()

            all_results.extend(response_data["RESULTSET"])
            next_position = response_data["NEXTPOSITION"]
            if next_position is None:
                break

            params["STARTPOSITION"] = next_position
            time.sleep(1)

    # 取得結果をデータフレームに変換
    df = pd.DataFrame(all_results)
    result_df = pd.DataFrame()

    for i in range(len(df)):
        values_df = pd.DataFrame(df.loc[i,"VALUES"])
        values_df = values_df.assign(
            SERIES_CODE=df.loc[i, "SERIES_CODE"],
            NAME_OF_TIME_SERIES_J=df.loc[i, "NAME_OF_TIME_SERIES_J"],
            CATEGORY_J=df.loc[i, "CATEGORY_J"]
        )
        result_df = pd.concat([result_df, values_df])

    result_df = result_df[["SERIES_CODE","NAME_OF_TIME_SERIES_J","CATEGORY_J","SURVEY_DATES","VALUES"]]

    result_df = result_df.reset_index(drop=True)

    result_df = result_df.astype({'SURVEY_DATES': str})

    return result_df

## 経常収支の推移と内訳を折れ線グラフと積み上げ棒グラフで表示しよう

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# データの取得
bop_series_codes_list = ["BPBP6JYNCB","BPBP6JYNTB","BPBP6JYNSN","BPBP6JYNPIN","BPBP6JYNSIN"]

bop_data_df = gen_boj_dataframe("BP01", bop_series_codes_list, "202001","202607")

# 値を偶数丸めする
bop_data_df = bop_data_df.round(0)

# データを経常収支とそれ以外に分割
bop_line_df = bop_data_df.query('SERIES_CODE == "BPBP6JYNCB"')
bop_bar_df = bop_data_df.query('SERIES_CODE != "BPBP6JYNCB"')

# 棒グラフで使うデータを横長形式に変換
bop_bar_wide_df = bop_bar_df.pivot(
    index="SURVEY_DATES",
    columns="NAME_OF_TIME_SERIES_J",
    values="VALUES"
).reset_index()

# グラフの作成
fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
    )

# 上段: 推移の折れ線
fig.add_trace(
    go.Scatter(
        x=bop_line_df["SURVEY_DATES"],
        y=bop_line_df["VALUES"],
        mode="lines+markers",
        name="経常収支"
        ),
    row=1,
    col=1,
    )

# 下段: 内訳の積み上げ棒グラフ
for col_name in bop_bar_df["NAME_OF_TIME_SERIES_J"].unique().tolist():
    fig.add_trace(
        go.Bar(
            x=bop_bar_wide_df["SURVEY_DATES"],
            y=bop_bar_wide_df[col_name],
            name=col_name
            ),
        row=2,
        col=1,
        )

fig.update_layout(
        title="経常収支の推移と内訳（億円）",
        yaxis_title="経常収支総額",
        yaxis2_title="経常収支内訳",
        hovermode="x unified",
        barmode="relative",
        legend_title="系列",
        )

# 上下段の縦軸とホバーの数値を3桁区切りの整数で表示
fig.update_yaxes(tickformat=",.0f", hoverformat=",.0f")

fig.show()